# Task 12: IVCAN Cable Inspection

**Student edition: Digital Image Processing and Deep Learning**

Implement cable and stripe segmentation for the IVCAN/Prysmian dataset. The notebook provides data loading, ground-truth masks, fixed five-fold splits, metrics, contract validation, and the final evaluator.

- Edit only the marked regions in `segmenta_cabo` and `segmenta_listra`. Initial baselines are supplied inside both regions.
- You may create helper functions, but do not change the function signatures, `metricas_segmentacao`, the folds, or the evaluator.
- Tune parameters only with each fold's training samples, record them in `parametros_por_fold`, and freeze them before the test fold.
- Inputs are **RGB**. Use explicit OpenCV RGB conversions such as `cv2.COLOR_RGB2HSV` or `cv2.COLOR_RGB2LAB`.
- Never access IDs, reference masks, ground truth, or external labels inside segmentation functions. `segmenta_listra` receives the predicted cable mask, never the real one.


## 1. Install and import libraries

In [ ]:
%pip install -q gdown

In [ ]:
import copy
import os
import zipfile
from pathlib import Path

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

plt.rcParams.update(
    {"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False}
)

## 2. Download and validate the public dataset

In [ ]:
# Public ZIP file containing the complete IVCAN dataset.
ZIP_ID = "1q0OG-pHmEKcgAML49jlaZTmn0suO1p2j"

ZIP_PATH = Path("/content/IVCAN-Dataset.zip")
EXTRACT_DIR = Path("/content/ivcan_zip_extracted")
DATASET_DIR = EXTRACT_DIR / "IVCAN-Dataset"
OUTPUT_DIR = Path("/content/ivcan_results")

# Download once per Colab session.
if not ZIP_PATH.is_file():
    result = gdown.download(
        id=ZIP_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )
    if not result:
        raise RuntimeError("Failed to download the public IVCAN ZIP.")

# Check whether the dataset has already been extracted completely.
def dataset_is_complete():
    if not all(
        (DATASET_DIR / name).exists()
        for name in ("dataset.csv", "Images", "Masks")
    ):
        return False

    images = DATASET_DIR / "Images"
    masks = DATASET_DIR / "Masks"

    return (
        sum(p.is_dir() for p in images.iterdir()) == 80
        and sum(p.is_dir() for p in masks.iterdir()) == 80
        and len(list(images.rglob("*.jpg"))) == 320
        and len(list(masks.rglob("*.jpg"))) == 552
    )

# Extract if needed.
if not dataset_is_complete():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        if archive.testzip() is not None:
            raise RuntimeError("The downloaded ZIP is corrupted.")
        archive.extractall(EXTRACT_DIR)

if not dataset_is_complete():
    raise RuntimeError(
        "Incomplete IVCAN dataset after extraction."
    )

# Preserve the original notebook interface.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pasta = str(DATASET_DIR)
resultados = OUTPUT_DIR

print("Dataset validated successfully!")
print("Dataset:", DATASET_DIR)
print("Images: 320")
print("Masks: 552")
print("Results:", OUTPUT_DIR)


## 3. Read dataset metadata

In [ ]:
# Dataset path was validated in the configuration cell.
caminho_csv = os.path.join(pasta, "dataset.csv")

# utf-8-sig trata o marcador BOM presente no arquivo.
dados = pd.read_csv(caminho_csv, encoding="utf-8-sig", na_values=["-"])

# Limpa os nomes das colunas e corrige o nome da cor da listra.
dados.columns = dados.columns.str.strip()
dados = dados.rename(columns={"Cor lista": "Cor listra"})

# Limpa os IDs e define essa coluna como índice.
dados["ID"] = dados["ID"].str.strip().str.upper()
dados = dados.set_index("ID", verify_integrity=True)


def consultar_id(dados, id_amostra):
    """
    Aceita, por exemplo: 1, "001", "ID001" ou "ID003_2".
    Imprime todas as informações e retorna uma Series do pandas.
    """
    texto = str(id_amostra).strip().upper().removeprefix("ID")
    numero, separador, experimento = texto.partition("_")
    id_normalizado = f"ID{int(numero):03d}"

    if separador:
        id_normalizado += f"_{int(experimento)}"

    if id_normalizado not in dados.index:
        raise KeyError(f"O ID {id_normalizado} não está presente no CSV.")

    informacoes = dados.loc[id_normalizado]

    print(f"\nAmostra: {id_normalizado}")
    print("-" * 55)

    for campo, valor in informacoes.items():
        valor_exibido = "Não informado / não se aplica" if pd.isna(valor) else valor
        print(f"{campo}: {valor_exibido}")

    return informacoes

In [ ]:
ID = "ID001"
print(dados.loc[ID])

## 4. Inspect images and masks

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def carregar_e_plotar_amostra(pasta_dataset, id_amostra, experimento=1):
    """
    id_amostra: inteiro ou string, como 1, "001" ou "ID001".
    experimento: 1 ou 2.

    Retorna:
        imagens:       array RGB com as quatro imagens.
        mascaras_cabo: array com as quatro máscaras binárias do cabo.
        mascaras_listra: array com as quatro máscaras binárias da listra.
        fig, axes: objetos do Matplotlib.
    """
    # Normaliza o ID.
    numero = int(str(id_amostra).strip().upper().removeprefix("ID"))

    if not 1 <= numero <= 40:
        raise ValueError("O ID deve estar entre 001 e 040.")

    if experimento not in (1, 2):
        raise ValueError("O experimento deve ser 1 ou 2.")

    amostra = f"ID{numero:03d}_{experimento}"
    pasta_dataset = Path(pasta_dataset)

    pasta_imagens = pasta_dataset / "Images" / amostra
    pasta_mascaras = pasta_dataset / "Masks" / amostra

    def listar_arquivos(pasta):
        if not pasta.is_dir():
            raise FileNotFoundError(f"Pasta não encontrada: {pasta}")

        # Permite tanto CAMERA_0.jpg quanto Camera_0.jpg.
        return {
            arquivo.name.lower(): arquivo
            for arquivo in pasta.iterdir()
            if arquivo.is_file()
        }

    arquivos_imagens = listar_arquivos(pasta_imagens)
    arquivos_mascaras = listar_arquivos(pasta_mascaras)

    def ler_imagem(arquivos, nome, pasta, mascara=False):
        caminho = arquivos.get(nome.lower())

        if caminho is None:
            raise FileNotFoundError(f"Arquivo não encontrado: {pasta / nome}")

        modo = cv2.IMREAD_GRAYSCALE if mascara else cv2.IMREAD_COLOR
        imagem = cv2.imread(str(caminho), modo)

        if imagem is None:
            raise RuntimeError(f"Não foi possível ler: {caminho}")

        if mascara:
            # Binariza para eliminar pequenas variações causadas pelo JPEG.
            return np.where(imagem > 127, 255, 0).astype(np.uint8)

        return cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

    imagens = []
    mascaras_cabo = []
    mascaras_listra = []

    for camera in range(4):
        imagem = ler_imagem(arquivos_imagens, f"CAMERA_{camera}.jpg", pasta_imagens)
        cabo = ler_imagem(
            arquivos_mascaras,
            f"CAMERA_{camera}_cable.jpg",
            pasta_mascaras,
            mascara=True,
        )

        nome_listra = f"CAMERA_{camera}_stripe.jpg"

        if nome_listra.lower() in arquivos_mascaras:
            listra = ler_imagem(
                arquivos_mascaras, nome_listra, pasta_mascaras, mascara=True
            )
        else:
            # Arquivo ausente representa um cabo sem listra.
            listra = np.zeros_like(cabo)

        if imagem.shape[:2] != cabo.shape or imagem.shape[:2] != listra.shape:
            raise ValueError(
                f"Câmera {camera}: imagem e máscaras possuem "
                f"dimensões diferentes: imagem={imagem.shape[:2]}, "
                f"cabo={cabo.shape}, listra={listra.shape}."
            )

        imagens.append(imagem)
        mascaras_cabo.append(cabo)
        mascaras_listra.append(listra)

    # Cada posição axes[linha, coluna] corresponde a um subplot.
    fig, axes = plt.subplots(4, 4, figsize=(18, 16))

    cor_cabo = (0, 255, 0)  # Verde em RGB.
    cor_listra = (255, 0, 0)  # Vermelho em RGB.

    for camera in range(4):
        imagem = imagens[camera]
        cabo = mascaras_cabo[camera]
        listra = mascaras_listra[camera]

        # Linha 1: imagens originais.
        axes[0, camera].imshow(imagem)
        axes[0, camera].set_title(f"Câmera {camera}", fontsize=13)

        # Linha 2: máscaras do cabo.
        axes[1, camera].imshow(cabo, cmap="gray", vmin=0, vmax=255)

        # Linha 3: máscaras da listra.
        axes[2, camera].imshow(listra, cmap="gray", vmin=0, vmax=255)

        # Linha 4: originais com contornos.
        sobreposicao = imagem.copy()

        for mascara, cor in ((cabo, cor_cabo), (listra, cor_listra)):
            contornos, _ = cv2.findContours(
                mascara.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            cv2.drawContours(
                sobreposicao, contornos, contourIdx=-1, color=cor, thickness=2
            )

        axes[3, camera].imshow(sobreposicao)

    nomes_linhas = [
        "Imagem original",
        "Máscara do cabo",
        "Máscara da listra",
        "Contornos",
    ]

    for linha in range(4):
        axes[linha, 0].set_ylabel(nomes_linhas[linha], fontsize=13)

        for coluna in range(4):
            ax = axes[linha, coluna]
            ax.set_xticks([])
            ax.set_yticks([])

            for borda in ax.spines.values():
                borda.set_visible(False)

    fig.suptitle(f"Amostra {amostra}", fontsize=18)

    fig.legend(
        handles=[
            Line2D([0], [0], color="lime", lw=2, label="Cabo"),
            Line2D([0], [0], color="red", lw=2, label="Listra"),
        ],
        loc="lower center",
        ncol=2,
        frameon=False,
        fontsize=12,
    )

    fig.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()

    return imagens, mascaras_cabo, mascaras_listra, fig, axes

In [ ]:
# The shared configuration cell already defines `pasta`.
ID = "ID010"
experimento = 1
carregar_e_plotar_amostra(pasta, ID, experimento)
print(dados.loc[ID])

## 5. Fixed five-fold split

In [ ]:
# Índice da lista: número do ID menos 1.
# Valor: fold de teste daquele ID e experimento.

folds_exp1 = [
    1,
    3,
    2,
    3,
    4,
    5,
    2,
    5,
    4,
    2,  # ID001–ID010
    4,
    5,
    4,
    2,
    1,
    2,
    3,
    1,
    3,
    1,  # ID011–ID020
    1,
    2,
    3,
    4,
    3,
    4,
    1,
    2,
    5,
    5,  # ID021–ID030
    3,
    5,
    4,
    1,
    1,
    2,
    3,
    5,
    5,
    4,  # ID031–ID040
]

folds_exp2 = [
    1,
    3,
    2,
    3,
    4,
    5,
    2,
    5,
    4,
    2,  # ID001–ID010
    4,
    5,
    4,
    2,
    1,
    2,
    3,
    2,
    3,
    1,  # ID011–ID020
    1,
    2,
    3,
    4,
    3,
    4,
    1,
    3,
    5,
    5,  # ID021–ID030
    4,
    5,
    4,
    1,
    1,
    2,
    3,
    5,
    5,
    1,  # ID031–ID040
]

# Chaves correspondem aos nomes das pastas.
fold_por_amostra = {
    f"ID{numero:03d}_{experimento}": lista[numero - 1]
    for experimento, lista in [(1, folds_exp1), (2, folds_exp2)]
    for numero in range(1, 41)
}

# Exemplo: fold 1 como teste; demais folds como treino.
fold_teste = 1

amostras_teste = [
    amostra for amostra, fold in fold_por_amostra.items() if fold == fold_teste
]

amostras_treino = [
    amostra for amostra, fold in fold_por_amostra.items() if fold != fold_teste
]

print("Teste:", amostras_teste)
print("Treino:", amostras_treino)

## 6. Provided metrics

In [ ]:
def _validar_mascaras(ground_truth, predicao):
    """Valida e converte duas máscaras binárias 2D para booleano."""
    gt = np.asarray(ground_truth)
    pred = np.asarray(predicao)

    if gt.ndim != 2 or pred.ndim != 2:
        raise ValueError("As máscaras devem ser 2D: (altura, largura).")

    if gt.shape != pred.shape:
        raise ValueError(
            f"As máscaras devem ter o mesmo tamanho: "
            f"ground truth={gt.shape}, predição={pred.shape}."
        )

    if gt.size == 0:
        raise ValueError("As máscaras não podem estar vazias.")

    for nome, mascara in [("ground_truth", gt), ("predicao", pred)]:
        if not np.all(np.isin(mascara, [0, 1, 255])):
            raise ValueError(
                f"{nome} deve conter apenas valores binários "
                "(False/True, 0/1 ou 0/255). "
                "Para probabilidades, aplique um limiar antes."
            )

    return gt != 0, pred != 0


def metricas_segmentacao(ground_truth, predicao, zero_division=np.nan):
    """
    Calcula métricas de segmentação binária por pixel.

    Parâmetros
    ----------
    ground_truth, predicao : array-like, shape (H, W)
        Máscaras binárias. Zero = fundo; valor não zero = objeto.
    zero_division : float, default np.nan
        Valor retornado quando o denominador de uma métrica é zero.

    Retorna
    -------
    dict
        Contagens TP, TN, FP, FN e métricas na escala de 0 a 1.
    """
    gt, pred = _validar_mascaras(ground_truth, predicao)

    tp = int(np.count_nonzero(gt & pred))
    tn = int(np.count_nonzero(~gt & ~pred))
    fp = int(np.count_nonzero(~gt & pred))
    fn = int(np.count_nonzero(gt & ~pred))

    def dividir(numerador, denominador):
        if denominador == 0:
            return float(zero_division)
        return numerador / denominador

    recall = dividir(tp, tp + fn)

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "IoU": dividir(tp, tp + fp + fn),
        "TPR": recall,
        "FPR": dividir(fp, fp + tn),
        "Precisao": dividir(tp, tp + fp),
        "Revocacao": recall,
        "Acuracia": dividir(tp + tn, tp + tn + fp + fn),
        "Dice_F1": dividir(2 * tp, 2 * tp + fp + fn),
        "Especificidade": dividir(tn, tn + fp),
    }


def visualizar_erros_segmentacao(ground_truth, predicao):
    """
    Gera uma imagem RGB uint8 de diagnóstico, com shape (H, W, 3).

    Preto:    TN — fundo correto.
    Verde:    TP — objeto correto.
    Vermelho: FP — falso alarme.
    Amarelo:  FN — objeto não detectado.

    A saída está em RGB, pronta para matplotlib.pyplot.imshow.
    """
    gt, pred = _validar_mascaras(ground_truth, predicao)

    imagem = np.zeros((*gt.shape, 3), dtype=np.uint8)

    imagem[gt & pred] = (0, 200, 0)  # TP: verde
    imagem[~gt & pred] = (255, 0, 0)  # FP: vermelho
    imagem[gt & ~pred] = (255, 220, 0)  # FN: amarelo
    # TN permanece preto.

    return imagem

### 6.1 Example: using the provided metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# 1. Criar duas imagens inicialmente pretas.
altura, largura = 300, 450

mascara_real = np.zeros((altura, largura), dtype=np.uint8)
mascara_predita = np.zeros_like(mascara_real)

# 2. Ground truth: retângulo e círculo preenchidos.
# No OpenCV, coordenadas são fornecidas como (x, y).
cv2.rectangle(mascara_real, (50, 60), (180, 230), 255, thickness=-1)
cv2.circle(mascara_real, (310, 150), 65, 255, thickness=-1)

# 3. Predição com erros intencionais.

# Retângulo deslocado: gera falsos positivos e falsos negativos.
cv2.rectangle(mascara_predita, (80, 80), (210, 250), 255, thickness=-1)

# Círculo menor e deslocado: deixa parte do objeto sem detectar.
cv2.circle(mascara_predita, (330, 150), 55, 255, thickness=-1)

# Objeto inexistente no ground truth: falso positivo.
cv2.circle(mascara_predita, (260, 45), 18, 255, thickness=-1)

# Buraco dentro do retângulo: falsos negativos.
cv2.circle(mascara_predita, (130, 150), 20, 0, thickness=-1)

# 4. Calcular métricas com a função anterior.
metricas = metricas_segmentacao(mascara_real, mascara_predita)

for nome, valor in metricas.items():
    if nome in {"TP", "TN", "FP", "FN"}:
        print(f"{nome:15s}: {valor}")
    else:
        print(f"{nome:15s}: {valor:.4f}")

# 5. Gerar imagem de diagnóstico com a função anterior.
diagnostico = visualizar_erros_segmentacao(mascara_real, mascara_predita)

# 6. Exibir máscaras e diagnóstico.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mascara_real, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Máscara real")

axes[1].imshow(mascara_predita, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Máscara predita")

axes[2].imshow(diagnostico)
axes[2].set_title(
    f"Diagnóstico\nIoU = {metricas['IoU']:.3f} | Dice = {metricas['Dice_F1']:.3f}"
)

for ax in axes:
    ax.axis("off")

fig.legend(
    handles=[
        Patch(facecolor="black", label="TN: fundo correto"),
        Patch(facecolor=(0, 200 / 255, 0), label="TP: objeto correto"),
        Patch(facecolor="red", label="FP: falso alarme"),
        Patch(facecolor=(1, 220 / 255, 0), label="FN: objeto não detectado"),
    ],
    loc="lower center",
    ncol=4,
)

plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()

## 7. Student segmentation

Implement only the two functions below. Return binary 2D masks with the same height and width as the input image; accepted values are bool, `0/1`, or `0/255`. The cable mask includes the stripe region.

Do not consult reference masks, sample IDs, stripe labels, or ground truth inside these functions. `segmenta_listra` must use the **predicted** cable mask and may return an all-zero mask when no stripe is visible.

In [ ]:
def segmenta_cabo(imagem_rgb, parametros=None):
    """Segmentação simples do cabo sobre fundo escuro."""
    parametros = {} if parametros is None else parametros
    mascara_cabo = np.zeros(imagem_rgb.shape[:2], dtype=np.uint8)

    ### START CODE HERE ###

    cinza = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)
    cinza = cv2.GaussianBlur(cinza, (5, 5), 0)

    # Sem limiar informado, utiliza Otsu.
    limiar = parametros.get("limiar", None)

    if limiar is None:
        _, binaria = cv2.threshold(cinza, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, binaria = cv2.threshold(cinza, float(limiar), 255, cv2.THRESH_BINARY)

    # Remove pequenos pontos e fecha pequenas descontinuidades.
    kernel = np.ones((3, 3), dtype=np.uint8)

    binaria = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, kernel)
    binaria = cv2.morphologyEx(binaria, cv2.MORPH_CLOSE, kernel)

    # Considera o maior contorno como o cabo e preenche seu interior.
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contornos:
        maior_contorno = max(contornos, key=cv2.contourArea)

        cv2.drawContours(
            mascara_cabo,
            [maior_contorno],
            contourIdx=-1,
            color=255,
            thickness=cv2.FILLED,
        )

    ### END CODE HERE ###

    return mascara_cabo

### Segment the stripe

Use the image and the **predicted** cable mask. The function must handle images without a visible stripe by returning an entirely black mask.

In [ ]:
def segmenta_listra(imagem_rgb, mascara_cabo_predita, parametros=None):
    """Detecta cores diferentes da cor predominante do cabo predito."""
    parametros = {} if parametros is None else parametros
    mascara_listra = np.zeros(imagem_rgb.shape[:2], dtype=np.uint8)

    ### START CODE HERE ###

    regiao_cabo = np.asarray(mascara_cabo_predita) > 0

    if np.any(regiao_cabo):
        suavizada = cv2.GaussianBlur(imagem_rgb, (3, 3), 0)

        # Conversão float: L em [0, 100]; a e b em unidades CIELab.
        lab = cv2.cvtColor(suavizada.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)

        # Aproxima a cor predominante pela mediana dos pixels do cabo.
        cor_referencia = np.median(lab[regiao_cabo], axis=0)

        diferenca = lab - cor_referencia

        # Reduz a influência de variações de iluminação.
        peso_l = float(parametros.get("peso_l", 0.5))
        diferenca[..., 0] *= peso_l

        distancia = np.linalg.norm(diferenca, axis=2)
        distancia_minima = float(parametros.get("distancia_minima", 18.0))

        candidata = regiao_cabo & (distancia > distancia_minima)
        mascara_listra = candidata.astype(np.uint8) * 255

        kernel = np.ones((3, 3), dtype=np.uint8)

        mascara_listra = cv2.morphologyEx(mascara_listra, cv2.MORPH_OPEN, kernel)
        mascara_listra = cv2.morphologyEx(mascara_listra, cv2.MORPH_CLOSE, kernel)

        # A morfologia pode expandir regiões; restringe ao cabo predito.
        mascara_listra[~regiao_cabo] = 0

    ### END CODE HERE ###

    return mascara_listra

## 8. Parameters and experimental protocol


O dicionário abaixo permite parâmetros diferentes por fold. As chaves internas
dependem da sua implementação: por exemplo, um limiar de intensidade ou o tamanho
de um elemento estruturante. Dicionários vazios usam os padrões que você definir.

Para o fold `k`, somente `fold != k` pode ser usado para escolher parâmetros.
`avaliar_cinco_folds` apenas avalia: não faz treinamento nem busca automaticamente.
Se houver uma etapa de ajuste, execute-a separadamente com as amostras de treino
retornadas por `separar_amostras_fold` e registre o resultado aqui.

As quatro câmeras de cada par ID–experimento permanecem no mesmo fold.
Nos IDs 018, 028, 031 e 040, os experimentos foram separados para manter exemplos
das cores exclusivas no treino. Nesses casos, a avaliação é entre experimentos
do mesmo cabo físico; registre essa limitação no relatório.


In [ ]:
def separar_amostras_fold(fold_por_amostra, fold_teste):
    """Retorna os nomes das pastas de treino e teste do fold escolhido."""
    if fold_teste not in (1, 2, 3, 4, 5):
        raise ValueError("fold_teste deve pertencer a {1, 2, 3, 4, 5}.")
    treino = sorted(a for a, f in fold_por_amostra.items() if f != fold_teste)
    teste = sorted(a for a, f in fold_por_amostra.items() if f == fold_teste)
    return treino, teste


parametros_por_fold = {fold: {"cabo": {}, "listra": {}} for fold in range(1, 6)}

# Exemplo de seleção de dados para desenvolvimento: treino do fold 1.
fold_desenvolvimento = 1
treino_desenvolvimento, teste_desenvolvimento = separar_amostras_fold(
    fold_por_amostra, fold_desenvolvimento
)
print("Pastas de treino:", treino_desenvolvimento)
print("Pastas de teste:", teste_desenvolvimento)

## 9. Provided evaluation loader


Este carregador não gera figuras durante a avaliação. Ele lê imagens em RGB e
binariza máscaras JPEG com limiar 127. Mantém a convenção já adotada no notebook:
arquivo de máscara de listra ausente representa ground truth vazio. Confirme que
essa ausência significa realmente cabo sem listra, e não anotação perdida.

Pasta/máscara do cabo ausente, JPEG truncado, dimensões incompatíveis e máscaras
uniformes incompatíveis com o cenário são erros de dados. O avaliador pode
interromper ou registrar e excluir esses casos, conforme a opção escolhida.
Ele não associa automaticamente `ID035_3` a `ID035_1` nem corrige anotações.


In [ ]:
import re


class ErroDadosAvaliacao(ValueError):
    """Erro nos arquivos de entrada; não representa falha do algoritmo discente."""


def _localizar_arquivo(pasta_arquivos, radical, obrigatorio=True):
    pasta_arquivos = Path(pasta_arquivos)
    if not pasta_arquivos.is_dir():
        raise ErroDadosAvaliacao(f"Pasta não encontrada: {pasta_arquivos}")
    extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    encontrados = [
        p
        for p in pasta_arquivos.iterdir()
        if p.is_file()
        and p.stem.lower() == radical.lower()
        and p.suffix.lower() in extensoes
    ]
    if len(encontrados) > 1:
        raise ErroDadosAvaliacao(
            f"Mais de um arquivo corresponde a {pasta_arquivos / radical}."
        )
    if not encontrados:
        if obrigatorio:
            raise ErroDadosAvaliacao(
                f"Arquivo não encontrado: {pasta_arquivos / radical}"
            )
        return None
    return encontrados[0]


def _ler_arquivo_verificado(caminho, modo):
    try:
        # O OpenCV pode aceitar JPEG truncado; exige-se decodificação completa.
        with Image.open(caminho) as imagem_pil:
            imagem_pil.load()
    except (OSError, ValueError) as erro:
        raise ErroDadosAvaliacao(f"Arquivo inválido: {caminho}: {erro}") from erro
    imagem = cv2.imread(str(caminho), modo)
    if imagem is None:
        raise ErroDadosAvaliacao(f"Não foi possível decodificar: {caminho}")
    return imagem


def carregar_imagem_avaliacao(pasta_dataset, amostra, camera):
    """Lê uma imagem RGB e duas máscaras, sem gerar figuras.

    Preserva a convenção do notebook: máscara de listra ausente = GT vazio.
    Uma pasta de máscaras ausente ou uma máscara de cabo ausente é erro.
    """
    if re.fullmatch(r"ID\d{3}_[12]", amostra) is None:
        raise ValueError(f"Nome inválido de amostra: {amostra}")
    if not 1 <= int(amostra[2:5]) <= 40 or camera not in range(4):
        raise ValueError("Esperados IDs 001 a 040 e câmeras 0 a 3.")
    raiz = Path(pasta_dataset)
    p_img = raiz / "Images" / amostra
    p_mask = raiz / "Masks" / amostra
    imagem_path = _localizar_arquivo(p_img, f"CAMERA_{camera}")
    cabo_path = _localizar_arquivo(p_mask, f"CAMERA_{camera}_cable")
    listra_path = _localizar_arquivo(
        p_mask, f"CAMERA_{camera}_stripe", obrigatorio=False
    )

    imagem_bgr = _ler_arquivo_verificado(imagem_path, cv2.IMREAD_COLOR)
    cabo = _ler_arquivo_verificado(cabo_path, cv2.IMREAD_GRAYSCALE)
    cabo = np.where(cabo > 127, 255, 0).astype(np.uint8)
    if listra_path is None:
        listra = np.zeros_like(cabo)
    else:
        listra = _ler_arquivo_verificado(listra_path, cv2.IMREAD_GRAYSCALE)
        listra = np.where(listra > 127, 255, 0).astype(np.uint8)
    if imagem_bgr.shape[:2] != cabo.shape or listra.shape != cabo.shape:
        raise ErroDadosAvaliacao(
            f"{amostra}, câmera {camera}: dimensões incompatíveis."
        )
    if not cabo.any() or np.all(cabo != 0) or np.all(listra != 0):
        raise ErroDadosAvaliacao(
            f"{amostra}, câmera {camera}: máscara uniforme incompatível "
            "com cabo sobre fundo."
        )
    return cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB), cabo, listra


def _validar_predicao_discente(predicao, shape, nome_funcao):
    if predicao is None:
        raise NotImplementedError(
            f"{nome_funcao} retornou None. Preencha a região "
            "START CODE HERE / END CODE HERE."
        )
    predicao = np.asarray(predicao)
    # Reaproveita a validação binária do notebook, sem passar nenhum GT discente.
    _, binaria = _validar_mascaras(np.zeros(shape, dtype=np.uint8), predicao)
    return binaria.astype(np.uint8) * 255

## 10. Contract test and a training-image visualization


Execute após implementar as duas funções. O teste verifica dimensões, valores
binários e preservação das entradas; ele não exige uma segmentação perfeita.
Uma função ainda não implementada gera `NotImplementedError`, evitando resultados
artificiais para máscaras vazias usadas como substitutas da implementação.


In [ ]:
def testar_contrato_segmentacao(
    imagem_rgb, parametros_cabo=None, parametros_listra=None
):
    """Testa a interface e a preservação das entradas; não mede a qualidade."""
    imagem_original = imagem_rgb.copy()
    pred_cabo = _validar_predicao_discente(
        segmenta_cabo(imagem_rgb, copy.deepcopy(parametros_cabo)),
        imagem_rgb.shape[:2],
        "segmenta_cabo",
    )
    np.testing.assert_array_equal(imagem_rgb, imagem_original)
    cabo_original = pred_cabo.copy()
    pred_listra = _validar_predicao_discente(
        segmenta_listra(imagem_rgb, pred_cabo, copy.deepcopy(parametros_listra)),
        imagem_rgb.shape[:2],
        "segmenta_listra",
    )
    np.testing.assert_array_equal(imagem_rgb, imagem_original)
    np.testing.assert_array_equal(pred_cabo, cabo_original)
    print(
        "Contrato verificado: máscaras binárias do tamanho correto; "
        "entradas preservadas."
    )
    return pred_cabo, pred_listra

In [ ]:
amostra_exemplo = treino_desenvolvimento[0]
camera_exemplo = 0
imagem_exemplo, gt_cabo_exemplo, gt_listra_exemplo = carregar_imagem_avaliacao(
    pasta, amostra_exemplo, camera_exemplo
)
p = parametros_por_fold[fold_desenvolvimento]
pred_cabo_exemplo, pred_listra_exemplo = testar_contrato_segmentacao(
    imagem_exemplo, p["cabo"], p["listra"]
)

fig, axes = plt.subplots(1, 5, figsize=(17, 6))
itens = [
    (imagem_exemplo, "Original RGB"),
    (pred_cabo_exemplo, "Cabo predito"),
    (pred_listra_exemplo, "Listra predita"),
    (visualizar_erros_segmentacao(gt_cabo_exemplo, pred_cabo_exemplo), "Erros: cabo"),
    (
        visualizar_erros_segmentacao(gt_listra_exemplo, pred_listra_exemplo),
        "Erros: listra",
    ),
]
for ax, (imagem_plot, titulo) in zip(axes, itens, strict=True):
    ax.imshow(imagem_plot, cmap="gray", vmin=0, vmax=255)
    ax.set_title(titulo)
    ax.axis("off")
fig.suptitle(
    f"Treino do fold {fold_desenvolvimento}: {amostra_exemplo}, câmera {camera_exemplo}"
)
plt.tight_layout()
plt.show()

## 11. Provided five-fold evaluation


Para cada imagem de teste, a função abaixo executa:

1. `segmenta_cabo(imagem_rgb, parametros_cabo)`;
2. `segmenta_listra(imagem_rgb, mascara_cabo_predita, parametros_listra)`;
3. `metricas_segmentacao` separadamente para cabo e listra.

As máscaras ground truth são usadas somente na avaliação. A previsão da listra
não recebe a máscara real do cabo e não é corrigida automaticamente pelo avaliador.

**Agregação:** cada métrica de um fold é a média das métricas de suas imagens
(média macro). A linha **Média** dá peso igual aos cinco folds. A linha
**Desvio-padrão** usa o desvio amostral entre folds, com `ddof=1`. TP, TN, FP e FN
são somados dentro de cada fold; suas linhas finais resumem esses totais por fold.
A coluna `N_imagens` informa o número efetivamente avaliado.

**Divisão por zero:** preserva-se `zero_division=np.nan` da função original.
Por exemplo, IoU e Dice são NaN quando GT e predição são ambos vazios; TPR é NaN
quando não há pixels positivos no GT. O pandas ignora NaNs nas médias, sem
substituí-los por 0 ou 1. As tabelas `valores_validos_por_fold` e
`folds_validos_no_resumo` mostram quantos valores contribuíram para cada média.
Se uma métrica tiver menos de dois folds válidos, seu desvio-padrão será NaN.

Sem exclusões, cada fold tem 16 pares ID–experimento e 64 imagens de teste.
Com exclusões, os tamanhos podem variar; um fold sem imagens válidas interrompe
a avaliação. Falhas no código discente não são tratadas como exclusões de dados.


In [ ]:
def avaliar_cinco_folds(
    pasta_dataset,
    fold_por_amostra,
    parametros_por_fold=None,
    *,
    zero_division=np.nan,
    ignorar_erros_dados=False,
    pasta_saida=None,
):
    """Avalia as duas funções discentes nos cinco testes externos.

    As métricas de cada fold são médias por imagem (macro), usando exatamente
    metricas_segmentacao. TP/TN/FP/FN são somas dos pixels dentro do fold.
    'Média' e 'Desvio-padrão' resumem os valores dos folds, com ddof=1.
    NaN é mantido nas métricas indefinidas e ignorado pelas médias do pandas.
    Os denominadores efetivos são retornados nas tabelas de valores válidos.

    Os parâmetros devem ter sido definidos apenas no treino de cada fold.
    Esta função não ajusta parâmetros. Erros na implementação do aluno sempre
    interrompem a execução, mesmo quando ignorar_erros_dados=True.
    """
    folds = [1, 2, 3, 4, 5]
    if set(fold_por_amostra.values()) != set(folds):
        raise ValueError("O mapeamento deve conter exatamente os folds 1, 2, 3, 4 e 5.")
    if parametros_por_fold is None:
        parametros_por_fold = {f: {"cabo": {}, "listra": {}} for f in folds}
    if set(parametros_por_fold) != set(folds):
        raise ValueError("Forneça parâmetros com chaves inteiras 1 a 5.")
    for f in folds:
        if not isinstance(parametros_por_fold[f], dict):
            raise TypeError(f"Os parâmetros do fold {f} devem ser um dicionário.")
        if set(parametros_por_fold[f]) != {"cabo", "listra"}:
            raise ValueError(f"Fold {f}: esperadas as chaves 'cabo' e 'listra'.")
        if not all(isinstance(v, dict) for v in parametros_por_fold[f].values()):
            raise TypeError(f"Fold {f}: 'cabo' e 'listra' devem conter dicionários.")

    registros, ocorrencias = [], []
    colunas_contagens = ["TP", "TN", "FP", "FN"]
    # Obtém os nomes da função existente, sem reimplementar suas fórmulas.
    exemplo = metricas_segmentacao(
        np.zeros((1, 1), np.uint8),
        np.zeros((1, 1), np.uint8),
        zero_division=zero_division,
    )
    colunas_metricas = [c for c in exemplo if c not in colunas_contagens]

    for fold in folds:
        _, amostras_teste = separar_amostras_fold(fold_por_amostra, fold)
        parametros = copy.deepcopy(parametros_por_fold[fold])
        n_avaliadas = 0
        for amostra in amostras_teste:
            for camera in range(4):
                try:
                    imagem_rgb, gt_cabo, gt_listra = carregar_imagem_avaliacao(
                        pasta_dataset, amostra, camera
                    )
                except ErroDadosAvaliacao as erro:
                    if not ignorar_erros_dados:
                        raise
                    ocorrencias.append(
                        {
                            "fold": fold,
                            "amostra": amostra,
                            "camera": camera,
                            "motivo": str(erro),
                        }
                    )
                    continue

                # Ground truth fica exclusivamente no avaliador.
                pred_cabo = _validar_predicao_discente(
                    segmenta_cabo(imagem_rgb.copy(), copy.deepcopy(parametros["cabo"])),
                    imagem_rgb.shape[:2],
                    "segmenta_cabo",
                )
                pred_listra = _validar_predicao_discente(
                    segmenta_listra(
                        imagem_rgb.copy(),
                        pred_cabo.copy(),
                        copy.deepcopy(parametros["listra"]),
                    ),
                    imagem_rgb.shape[:2],
                    "segmenta_listra",
                )
                # Nenhuma correção, recorte ou interseção com GT da previsão.
                for alvo, gt, pred in [
                    ("Cabo", gt_cabo, pred_cabo),
                    ("Listra", gt_listra, pred_listra),
                ]:
                    valores = metricas_segmentacao(
                        gt, pred, zero_division=zero_division
                    )
                    registros.append(
                        {
                            "alvo": alvo,
                            "fold": fold,
                            "amostra": amostra,
                            "camera": camera,
                            "gt_vazio": not np.any(gt),
                            "pred_vazia": not np.any(pred),
                            **valores,
                        }
                    )
                n_avaliadas += 1

        if n_avaliadas == 0:
            raise ValueError(
                f"Fold {fold} sem imagens válidas: não é possível resumir cinco folds."
            )
        print(f"Fold {fold}: {n_avaliadas} imagens de teste avaliadas.")

    por_imagem = pd.DataFrame(registros)
    grupos = por_imagem.groupby(["alvo", "fold"], sort=True)
    taxas = grupos[colunas_metricas].mean()
    contagens = grupos[colunas_contagens].sum()
    tamanhos = grupos.size().rename("N_imagens")
    por_fold = pd.concat([tamanhos, contagens, taxas], axis=1)
    validos_por_fold = grupos[colunas_metricas].count()

    # Cada fold tem o mesmo peso na média final, mesmo se houve exclusões.
    media = por_fold.groupby(level="alvo").mean()
    desvio = por_fold.groupby(level="alvo").std(ddof=1)
    blocos = []
    for alvo in ["Cabo", "Listra"]:
        bloco = por_fold.loc[alvo].copy()
        bloco.index = [f"Fold {f}" for f in bloco.index]
        bloco.loc["Média"] = media.loc[alvo]
        bloco.loc["Desvio-padrão"] = desvio.loc[alvo]
        bloco.index.name = "Resultado"
        blocos.append(bloco)
    tabela = pd.concat(blocos, keys=["Cabo", "Listra"], names=["Alvo", "Resultado"])
    resumo = pd.concat(
        {"Média": media, "Desvio-padrão": desvio}, names=["Estatística", "Alvo"]
    )
    folds_validos = por_fold[colunas_metricas].groupby(level="alvo").count()
    erros = pd.DataFrame(ocorrencias, columns=["fold", "amostra", "camera", "motivo"])

    resultado = {
        "tabela": tabela,
        "por_imagem": por_imagem,
        "por_fold": por_fold,
        "resumo": resumo,
        "valores_validos_por_fold": validos_por_fold,
        "folds_validos_no_resumo": folds_validos,
        "erros_dados": erros,
    }
    if pasta_saida is not None:
        pasta_saida = Path(pasta_saida)
        pasta_saida.mkdir(parents=True, exist_ok=True)
        for nome, df in resultado.items():
            df.to_csv(
                pasta_saida / f"{nome}.csv",
                encoding="utf-8-sig",
                index=nome not in {"por_imagem", "erros_dados"},
            )
    return resultado

### Run after freezing the parameters of all five folds


O exemplo usa `ignorar_erros_dados=True` para registrar os problemas conhecidos
da base e avaliar os arquivos válidos. Para exigir a base integralmente válida,
altere para `False`. Informe as exclusões no relatório; não apresente os resultados
de um subconjunto como se todas as 320 imagens tivessem sido avaliadas.


In [ ]:
# avaliacao = avaliar_cinco_folds(
#     pasta_dataset=pasta,
#     fold_por_amostra=fold_por_amostra,
#     parametros_por_fold=parametros_por_fold,
#     zero_division=np.nan,
#     ignorar_erros_dados=True,
#     pasta_saida=resultados / "avaliacao_discente",
# )
avaliacao = avaliar_cinco_folds(
    pasta_dataset=pasta,
    fold_por_amostra=fold_por_amostra,
    parametros_por_fold=parametros_por_fold,
    ignorar_erros_dados=True,
    pasta_saida=resultados / "avaliacao_discente",
)

display(avaliacao["tabela"].style.format(precision=4, na_rep="Não definido"))


# Tabela completa: cinco folds, média e desvio-padrão para Cabo e Listra.
# display(avaliacao["tabela"].style.format(precision=4, na_rep="Não definido"))

if not avaliacao["erros_dados"].empty:
    print("Imagens excluídas por problemas nos dados:")
    display(avaliacao["erros_dados"])

print("Número de folds com valor definido em cada métrica:")
display(avaliacao["folds_validos_no_resumo"])

# Também disponíveis:
# avaliacao["por_imagem"]                  métricas de cada imagem/câmera/alvo;
# avaliacao["por_fold"]                    resultados numéricos dos folds;
# avaliacao["resumo"]                      média e desvio-padrão;
# avaliacao["valores_validos_por_fold"]     número de imagens por média de métrica.

## 12. Results discussion


* Por que a acurácia pode ser alta mesmo quando a IoU da listra é baixa?
* Como seu método decide que uma imagem não possui listra visível?
* Quais cores, sombras ou reflexos provocam os maiores erros?
* A passagem de um cabo predito para a segmentação da listra propaga quais erros?
* Quais parâmetros foram ajustados em cada treino e como foram selecionados?
* Como exclusões, métricas indefinidas e a separação dos quatro IDs de cores
  exclusivas limitam a interpretação das médias dos folds?
